In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, mean_absolute_error, accuracy_score
from collections import Counter
from imblearn.over_sampling import SMOTE
import joblib

# -------------------- Constants & Helper Functions -------------------- #
TEAM_MAPPING = {
    'Man Utd': 'Manchester Utd', 'Man United': 'Manchester Utd',
    'Man City': 'Manchester City', 'Newcastle Utd': 'Newcastle United',
    'Newcastle Ut': 'Newcastle United', "Nott'ham Forest": 'Nottingham Forest',
    'Paris S-G': 'Paris Saint-Germain', 'Inter Milan': 'Inter',
    'Spurs': 'Tottenham', 'West Ham Utd': 'West Ham United'
}

def clean_numeric_values(value):
    if isinstance(value, str):
        cleaned = value.replace(',', '').replace(' ', '')
        if cleaned.replace('.', '', 1).isdigit():
            return float(cleaned)
    return value

def standardize_team_names(df, column_name):
    df = df.copy()
    df.loc[:, column_name] = (
        df[column_name]
        .replace(TEAM_MAPPING)
        .str.strip()
    )
    return df

# -------------------- Model Architecture -------------------- #
class FootballPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.shared_base = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.class_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 3)
        )
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 2)
        )

    def forward(self, x):
        x = self.shared_base(x)
        return self.class_head(x), self.reg_head(x)

# -------------------- Training Utilities -------------------- #
class EarlyStopper:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_val_loss = float('inf')

    def __call__(self, val_loss):
        if val_loss < self.min_val_loss - self.min_delta:
            self.min_val_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

def load_and_preprocess_data():
    stats_df = pd.read_csv('Combined_Leagues_Stats.csv').copy()
    fixtures_df = pd.read_csv('Fixture_Results.csv').copy()

    numeric_cols = ['progressive_carries', 'progressive_passes', 'xg', 'npxg',
                   'xg_assist', 'npxg_xg_assist', 'goals_per90', 'assists_per90',
                   'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
                   'xg_assist_per90', 'npxg_per90', 'Home_xG', 'Away_xG']

    for df in [stats_df, fixtures_df]:
        for col in numeric_cols:
            if col in df.columns:
                df.loc[:, col] = df[col].apply(clean_numeric_values)

    # Drop rows with missing target values
    fixtures_df = fixtures_df.dropna(subset=['Home_xG', 'Away_xG']).copy()

    # Standardize team names
    stats_df = standardize_team_names(stats_df, 'team')
    fixtures_df = standardize_team_names(fixtures_df, 'Home_Team')
    fixtures_df = standardize_team_names(fixtures_df, 'Away_Team')
    
    # Create features
    stats_df.loc[:, 'total_progression'] = (
        stats_df['progressive_carries'] + stats_df['progressive_passes']
    )

    feature_columns = [
        'total_progression', 'goals_per90', 'assists_per90',
        'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
        'xg_assist_per90', 'npxg_per90'
    ]

    # Impute features
    imputer = SimpleImputer(strategy='median')
    stats_df.loc[:, feature_columns] = imputer.fit_transform(stats_df[feature_columns])
    
    team_stats = stats_df.set_index('team')[feature_columns].to_dict('index')
    fixtures_df = fixtures_df.loc[fixtures_df['Home_Team'] != fixtures_df['Away_Team']].copy()
    
    return team_stats, fixtures_df, feature_columns

# -------------------- Training Execution -------------------- #
if __name__ == "__main__":
    team_stats, fixtures_df, feature_columns = load_and_preprocess_data()
    
    def get_features(row):
        home_team = row['Home_Team'].strip()
        away_team = row['Away_Team'].strip()
        home_features = list(team_stats.get(home_team, {}).values()) or [0]*len(feature_columns)
        away_features = list(team_stats.get(away_team, {}).values()) or [0]*len(feature_columns)
        return home_features + away_features

    fixtures_df.loc[:, 'features'] = fixtures_df.apply(get_features, axis=1)
    fixtures_df.loc[:, 'result'] = fixtures_df.apply(
        lambda row: 2 if row['Home_Score'] > row['Away_Score'] else 1 if row['Home_Score'] == row['Away_Score'] else 0, 
        axis=1
    )

    X = np.array(fixtures_df['features'].tolist())
    y_class = fixtures_df['result'].values
    y_reg = fixtures_df[['Home_xG', 'Away_xG']].values
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    best_models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_class_train, y_class_val = y_class[train_idx], y_class[val_idx]
        y_reg_train, y_reg_val = y_reg[train_idx], y_reg[val_idx]

        # Handle class imbalance
        smote = SMOTE(random_state=42)
        X_train_res, y_class_train_res = smote.fit_resample(X_train, y_class_train)
        y_reg_train_res = y_reg_train[np.arange(len(y_class_train_res)) % len(y_reg_train)]

        # Normalization
        scaler = StandardScaler()
        X_train_res = scaler.fit_transform(X_train_res)
        X_val = scaler.transform(X_val)

        # Save artifacts from first fold
        if fold == 0:
            joblib.dump({
                'team_stats': team_stats,
                'feature_columns': feature_columns,
                'scaler': scaler
            }, 'model_artifacts.pkl')

        class FootballDataset(Dataset):
            def __init__(self, X, y_class, y_reg):
                self.X = torch.tensor(X, dtype=torch.float32)
                self.y_class = torch.tensor(y_class, dtype=torch.long)
                self.y_reg = torch.tensor(y_reg, dtype=torch.float32)
            
            def __len__(self): return len(self.X)
            def __getitem__(self, idx): return self.X[idx], self.y_class[idx], self.y_reg[idx]

        train_dataset = FootballDataset(X_train_res, y_class_train_res, y_reg_train_res)
        val_dataset = FootballDataset(X_val, y_class_val, y_reg_val)

        # Model setup
        model = FootballPredictor(X_train_res.shape[1])
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
        early_stopper = EarlyStopper(patience=8)
        
        class_weights = torch.tensor([1/(count+1e-5) for count in np.bincount(y_class_train_res)], dtype=torch.float32)
        criterion_class = nn.CrossEntropyLoss(weight=class_weights)
        criterion_reg = nn.HuberLoss()

        best_val_loss = float('inf')
        for epoch in range(100):
            model.train()
            total_loss = 0
            for X_batch, y_class_batch, y_reg_batch in DataLoader(train_dataset, batch_size=32, shuffle=True):
                optimizer.zero_grad()
                class_out, reg_out = model(X_batch)
                loss = 0.7*criterion_class(class_out, y_class_batch) + 0.3*criterion_reg(reg_out, y_reg_batch)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            # Validation
            model.eval()
            val_loss, val_mae = 0, 0
            all_true, all_pred = [], []
            with torch.no_grad():
                for X_batch, y_class_batch, y_reg_batch in DataLoader(val_dataset, batch_size=32):
                    class_out, reg_out = model(X_batch)
                    val_loss += 0.7*criterion_class(class_out, y_class_batch) + 0.3*criterion_reg(reg_out, y_reg_batch)
                    
                    # Collect predictions for accuracy calculation
                    preds = class_out.argmax(dim=1).numpy()
                    all_true.extend(y_class_batch.numpy())
                    all_pred.extend(preds)
                    
                    val_mae += mean_absolute_error(y_reg_batch.numpy(), reg_out.numpy())

            # Calculate metrics
            avg_val_loss = val_loss / len(val_dataset)
            val_acc = accuracy_score(all_true, all_pred)
            val_f1 = f1_score(all_true, all_pred, average='macro')
            avg_val_mae = val_mae / len(val_dataset)

            print(f"Fold {fold+1} Epoch {epoch+1}: "
                  f"Train Loss: {total_loss/len(train_dataset):.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} | "
                  f"Val Acc: {val_acc:.4f} | "
                  f"Val F1: {val_f1:.4f} | "
                  f"Val MAE: {avg_val_mae:.4f}")
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), f'best_model_fold{fold+1}.pth')
            
            if early_stopper(avg_val_loss):
                print(f"Early stopping triggered at fold {fold+1}")
                break
        
        best_models.append(model)

    # Save final ensemble model
    torch.save([model.state_dict() for model in best_models], 'ensemble_models.pth')

C:\Users\DATA-JOHN\AppData\Local\Temp\ipykernel_6052\1303384664.py:123: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  stats_df.loc[:, feature_columns] = imputer.fit_transform(stats_df[feature_columns])


Fold 1 Epoch 1: Train Loss: 0.0260 | Val Loss: 0.0260 | Val Acc: 0.4860 | Val F1: 0.4682 | Val MAE: 0.0186
Fold 1 Epoch 2: Train Loss: 0.0247 | Val Loss: 0.0257 | Val Acc: 0.4836 | Val F1: 0.4588 | Val MAE: 0.0184
Fold 1 Epoch 3: Train Loss: 0.0246 | Val Loss: 0.0257 | Val Acc: 0.4860 | Val F1: 0.4402 | Val MAE: 0.0183
Fold 1 Epoch 4: Train Loss: 0.0245 | Val Loss: 0.0258 | Val Acc: 0.4743 | Val F1: 0.4610 | Val MAE: 0.0183
Fold 1 Epoch 5: Train Loss: 0.0246 | Val Loss: 0.0257 | Val Acc: 0.4790 | Val F1: 0.4662 | Val MAE: 0.0183
Fold 1 Epoch 6: Train Loss: 0.0245 | Val Loss: 0.0256 | Val Acc: 0.4696 | Val F1: 0.4432 | Val MAE: 0.0182
Fold 1 Epoch 7: Train Loss: 0.0243 | Val Loss: 0.0257 | Val Acc: 0.4860 | Val F1: 0.4567 | Val MAE: 0.0183
Fold 1 Epoch 8: Train Loss: 0.0244 | Val Loss: 0.0255 | Val Acc: 0.5000 | Val F1: 0.4709 | Val MAE: 0.0180
Fold 1 Epoch 9: Train Loss: 0.0243 | Val Loss: 0.0256 | Val Acc: 0.4907 | Val F1: 0.4603 | Val MAE: 0.0183
Early stopping triggered at fold 1
Fo

In [2]:
# -------------------- Prediction Function -------------------- #
def predict_fixtures(fixtures_csv_path):
    artifacts = joblib.load('model_artifacts.pkl')
    team_stats = artifacts['team_stats']
    feature_columns = artifacts['feature_columns']
    scaler = artifacts['scaler']
    
    model = FootballPredictor(len(feature_columns)*2)
    model.load_state_dict(torch.load('best_model_fold1.pth', weights_only=True))
    model.eval()
    
    new_fixtures = pd.read_csv(fixtures_csv_path).copy()
    new_fixtures = standardize_team_names(new_fixtures, 'Home_Team')
    new_fixtures = standardize_team_names(new_fixtures, 'Away_Team')
    
    def get_features(row):
        home_team = row['Home_Team'].strip()
        away_team = row['Away_Team'].strip()
        home_features = list(team_stats.get(home_team, {}).values()) or [0]*len(feature_columns)
        away_features = list(team_stats.get(away_team, {}).values()) or [0]*len(feature_columns)
        return home_features + away_features
    
    new_fixtures.loc[:, 'features'] = new_fixtures.apply(get_features, axis=1)
    X_new = scaler.transform(np.array(new_fixtures['features'].tolist()))
    X_new = torch.tensor(X_new, dtype=torch.float32)
    
    with torch.no_grad():
        class_logits, reg_preds = model(X_new)
        class_probs = torch.softmax(class_logits, dim=1).numpy()
    
    return pd.DataFrame({
        'Home_Team': new_fixtures['Home_Team'],
        'Away_Team': new_fixtures['Away_Team'],
        'Home_Win_Prob': class_probs[:, 2],
        'Draw_Prob': class_probs[:, 1],
        'Away_Win_Prob': class_probs[:, 0],
        'Predicted_Home_xG': reg_preds[:, 0].numpy(),
        'Predicted_Away_xG': reg_preds[:, 1].numpy()
    })

In [3]:
# Generate predictions
predictions = predict_fixtures('fixtures.csv')

# Sort by probability difference
predictions['Prob_Diff'] = abs(predictions['Home_Win_Prob'] - predictions['Away_Win_Prob'])
final_predictions = predictions.sort_values('Prob_Diff', ascending=False)

final_predictions

,Home_Team,Away_Team,Home_Win_Prob,Draw_Prob,Away_Win_Prob,Predicted_Home_xG,Predicted_Away_xG,Prob_Diff
5,Liverpool,Everton,0.808846,0.150818,0.040336,2.474586,0.697733,0.768510
2,Manchester City,Leicester City,0.741653,0.205913,0.052434,1.965269,0.506180,0.689219
0,Bournemouth,Ipswich Town,0.663261,0.233822,0.102917,1.950870,0.643956,0.560344
7,Atlético Madrid,Barcelona,0.090263,0.366973,0.542764,0.858913,1.487431,0.452500
4,Southampton,Crystal Palace,0.134279,0.289555,0.576166,1.111300,1.450153,0.441887
6,Milan,Inter,0.174615,0.359489,0.465896,1.016950,1.054356,0.291280
3,Newcastle United,Brentford,0.341857,0.321464,0.336679,1.483518,1.138149,0.005178
1,Brighton,Aston Villa,0.325588,0.352718,0.321694,1.593624,1.252526,0.003894
